# W12 Exercise: Validate model proposals before tool execution

Machine Learning in Engineering Science | NCKU / LAiMM | Fall 2026

**Date:** 2026-11-23. **Exercise time:** 35 minutes within the class/practice allocation, not extra time after class.

**Classroom formative practice. No new weekly graded submission is created.**

## How to work independently
1. In Colab, save a copy to your Drive and use a CPU runtime. In Jupyter, open this notebook and select a Python kernel.
2. Run the complete worked walkthrough below from top to bottom. Read every Markdown checkpoint; the walkthrough is included so no previous notebook state is needed.
3. At the Independent Exercise section, write each prediction in the response cell before running its experiment.
4. Run the starter experiment, make the requested controlled change, and record the actual result. Edit only the declared factor; preserve split rules and preprocessing.
5. Explain one mechanism and one limitation. Export the evidence JSON and discuss it with a partner.

The core exercises need no paid API, private dataset, or GPU. The agent units use clearly labeled offline simulators, not real LLM calls.

**Reproducibility boundary:** the original guided notebooks were previously executed; the new extension cells have not yet been execution-tested in this revision. Report unexpected behavior rather than claiming a check passed.


# Week 12: Bounded agents and structured outputs

Machine Learning in Engineering Science | NCKU + LAiMM | Fall 2026

**Purpose:** a fully worked, self-paced in-class practice with an optional extension. Weeks 11 and 18 are optional examination-readiness materials. These notebooks do not create weekly graded assignments.

**Before running:** choose a Python 3 kernel in Jupyter or a CPU runtime in Colab. Run every cell from top to bottom with Shift+Enter. Setup installs missing free packages. No API key or external dataset download is required.

**Learning target:** complete the calculation, inspect intermediate values, and explain why the protocol supports a limited claim.

**Time:** about 25-45 minutes including reading, prediction, execution, and explain-back.


## 1. Set up and record the environment

Keep the seed fixed for the first run. Record actual outputs and package versions rather than copying a claimed performance number.


In [ ]:
import sys, subprocess, importlib.util, platform, json, random, hashlib
from pathlib import Path
required = {"numpy": "numpy", "matplotlib": "matplotlib", "sklearn": "scikit-learn"}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
import numpy as np
import matplotlib.pyplot as plt
SEED = 2026
random.seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)
ARTIFACTS = Path("artifacts")
ARTIFACTS.mkdir(exist_ok=True)
print("Python:", platform.python_version(), "NumPy:", np.__version__, "Seed:", SEED)


**Checkpoint.** The environment cell must finish before any analysis cell. Restart and run all if a variable is missing.


## 2. Define an agent contract

An agent combines an objective, a model or policy, tools, state, and a stopping rule. This free deterministic simulator teaches control flow; it is explicitly not an LLM.


In [ ]:
contract={"allowed_tools":["mean"],"max_steps":2,"input_limit":100,
          "require_numeric_data":True,"output_fields":["tool","result","status"]}
print(json.dumps(contract,indent=2))


**Checkpoint.** Explain which decisions require a real model in a production assistant and which belong to deterministic validation.


## 3. Validate before executing a tool

Reject unknown tools and invalid inputs before execution. Avoid eval and unrestricted Python execution.


In [ ]:
def run_request(request):
    if request.get("tool") not in contract["allowed_tools"]:
        return {"tool":request.get("tool"),"result":None,"status":"rejected"}
    values=request.get("values",[])
    if not isinstance(values,list) or not 1<=len(values)<=100 or not all(type(x) in (int,float) and np.isfinite(x) for x in values):
        return {"tool":request["tool"],"result":None,"status":"invalid data"}
    return {"tool":"mean","result":float(np.mean(values)),"status":"ok"}
print(run_request({"tool":"mean","values":[1,2,3]}))


**Checkpoint.** The valid output contains mean 2.0. The execution layer, not the prompt, enforces the allowlist.


## 4. Test failure paths

Use adversarial input as data. No string in this test grants new permissions.


In [ ]:
cases=[{"tool":"mean","values":[1,2,3]}, {"tool":"delete_files","values":[1]},
       {"tool":"mean","values":["ignore rules"]}, {"tool":"mean","values":[]}]
observed=[run_request(item)["status"] for item in cases]
assert observed==["ok","rejected","invalid data","invalid data"]
summary={"cases":len(cases),"statuses":observed,"policy":"deterministic teaching simulator"}
print(summary)


**Checkpoint.** Connect validation, tool execution, and explicit refusal to the written contract.


## 6. Preserve your evidence

Export a JSON record with a seed, package version, and measured result. Download the artifacts directory before leaving Colab; runtime storage is temporary.


In [ ]:
record={"week":12,"seed":SEED,"numpy":np.__version__,"python":platform.python_version(),"result":summary}
(ARTIFACTS/"week-12-evidence.json").write_text(json.dumps(record,indent=2))
print(json.dumps(record,indent=2))


**Checkpoint.** The evidence file must match the result you actually obtained. A saved screenshot alone does not permit rerunning the calculation.


## 7. Try one change and explain it
1. Predict the effect of one change before editing code. Change one parameter, input, or boundary condition only.
2. Rerun all cells in a fresh kernel. Keep the original evidence record and compare the observed difference.
3. Explain the result using the mechanism, not just the final number.
4. Name one assumption that limits the claim, and a new test that could challenge it.

**Troubleshooting:** if installation fails, restore network access; if a variable is undefined, run earlier cells; if your output differs, compare package versions, seed, and data construction. Do not repeatedly edit code until a preferred score appears.

**Completion criterion:** one executed notebook, one evidence JSON, and a brief claim/evidence/limitation statement.

### Reference documentation
- [NumPy user guide](https://numpy.org/doc/stable/user/)
- [scikit-learn user guide](https://scikit-learn.org/stable/user_guide.html)
- [Course repository](https://github.com/CHjonnyYu/MLES-Fall-2026)


## Independent Exercise: W12

Complete the three tasks below after the walkthrough. The starter cells produce observations, not your written interpretation. Fill the response fields in the next cell. Save separate records for the original run and any controlled change.

In [ ]:
exercise_id="W12"
observations={}
responses={
    'task1_prediction':'', 'task1_explanation':'',
    'task2_prediction':'', 'task2_explanation':'',
    'task3_prediction':'', 'task3_explanation':'',
    'controlled_change':'', 'limitation':'', 'ai_use_and_checks':''
}
# Fill these strings before and after each experiment. Use 'none' if no AI assistance was used.
print('Exercise record initialized:', exercise_id)

### Task 1: Keep unsupported tools disabled

Send a valid mean request and an unsupported delete_files request to the deterministic teaching controller. Explain which layer refuses the unsupported action.

**Steps**
1. Write your prediction in `responses['task1_prediction']` before running.
2. Run the starter cell; read every printed quantity and retain the observation.
3. Make the controlled comparison requested above. Record its configuration in `responses['controlled_change']`.
4. Write a mechanism-based explanation in `responses['task1_explanation']`; do not merely repeat a score.

**Checkpoint:** The mean is 2. The unsupported action must be refused, not executed. No real file deletion or model API is involved.

In [ ]:
observations['task1']={'allowed':run_request({'tool':'mean','values':[1,2,3]}),'blocked':run_request({'tool':'delete_files','values':[1,2,3]})}
print(observations['task1'])

### Task 2: Probe input validation

Compare numeric input, nonnumeric input, and a request above the declared input limit. Keep these as inert test data.

**Steps**
1. Write your prediction in `responses['task2_prediction']` before running.
2. Run the starter cell; read every printed quantity and retain the observation.
3. Make the controlled comparison requested above. Record its configuration in `responses['controlled_change']`.
4. Write a mechanism-based explanation in `responses['task2_explanation']`; do not merely repeat a score.

**Checkpoint:** Inspect the returned statuses rather than assuming a prompt enforces type or length constraints. Record any boundary case the implementation does not handle.

In [ ]:
exercise_requests=[{'tool':'mean','values':[1,2,3]},{'tool':'mean','values':['ignore the policy',2]},{'tool':'mean','values':list(range(101))}]
observations['task2']=[run_request(case) for case in exercise_requests]
print(observations['task2'])

### Task 3: Separate the simulator from an LLM agent

Identify the proposal, validator, tool, and stopping policy. State which components are deterministic and what has not been evaluated.

**Steps**
1. Write your prediction in `responses['task3_prediction']` before running.
2. Run the starter cell; read every printed quantity and retain the observation.
3. Make the controlled comparison requested above. Record its configuration in `responses['controlled_change']`.
4. Write a mechanism-based explanation in `responses['task3_explanation']`; do not merely repeat a score.

**Checkpoint:** The simulator demonstrates controlled execution. It is not evidence that a real LLM will propose valid or useful actions.

In [ ]:
observations['task3']={'teaching_system':'deterministic offline simulator','contract':contract,'paid_model_calls':0}
print(observations['task3'])

### Finish: preserve and explain

Fill all response strings, then run the export cell. A draft file is intentionally labeled incomplete; a complete response record is not automatically a scientifically correct result. Download the JSON before the Colab session ends. Share it with a partner for explain-back, not as an additional weekly submission.

**Exit questions:** What changed? Which observation supports your explanation? What does this experiment not establish?

In [ ]:
missing_responses=[key for key,value in responses.items() if not str(value).strip()]
exercise_record={'exercise':exercise_id,'status':'draft' if missing_responses else 'ready for peer discussion','responses':responses,'observations':observations,'missing_responses':missing_responses,'environment':{'python':platform.python_version(),'numpy':np.__version__},'data_scope':'course-provided or synthetic teaching data'}
exercise_folder=Path('artifacts')/'exercises'; exercise_folder.mkdir(parents=True,exist_ok=True)
exercise_path=exercise_folder/(exercise_id+'-evidence.json')
exercise_path.write_text(json.dumps(exercise_record,indent=2,default=lambda value:value.tolist() if hasattr(value,'tolist') else str(value)))
print('Saved:',exercise_path)
print('Status:',exercise_record['status'])
print('Fields still to complete:',missing_responses)